In [16]:
import cv2
import os
import pandas as pd

In [17]:
def encontrar_esquinas(img):
    detector = cv2.FastFeatureDetector_create(
        threshold=60,
        nonmaxSuppression=True,
        type=cv2.FastFeatureDetector_TYPE_7_12
    )
    return detector.detect(img, mask=None)

def dibujar_puntos(img, pts):
    return cv2.drawKeypoints(img, pts, None, color=(0, 0, 255))

In [18]:
def procesar_telas():
    path = "img/telas"
    files = os.listdir(path)

    resultados = []

    for file in files:
        ruta = os.path.join(path, file)

        img = cv2.imread(ruta)
        if img is None:
            continue

        img_gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

        pts = encontrar_esquinas(img_gray)

        # Cálculo de densidad
        num_esquinas = len(pts)
        alto, ancho = img_gray.shape
        total_pixeles = alto * ancho

        densidad = num_esquinas / total_pixeles

        resultados.append({
            "Tela": file,
            "Esquinas": num_esquinas,
            "Pixeles": total_pixeles,
            "Densidad": densidad
        })

    return resultados

In [19]:
resultados = procesar_telas()

tabla = pd.DataFrame(resultados)

# Ordenar por densidad (de mayor a menor)
tabla = tabla.sort_values(by="Densidad", ascending=False)

tabla

,Tela,Esquinas,Pixeles,Densidad
6,Oxford.jpg,11650,300000,0.038833
11,Terry.jpg,3369,300000,0.011230
7,Polyester.jpg,2378,300000,0.007927
12,Tweed.jpg,281,300000,0.000937
10,Silk.jpg,215,300000,0.000717
3,Faux-Fur.jpg,24,300000,0.000080
5,Linen.jpg,8,300000,0.000027
0,Corduroy.jpg,3,300000,0.000010
1,Cotton-Flannel.jpg,1,300000,0.000003
2,Cotton-Jersey.jpg,1,300000,0.000003


In [20]:
seleccion = tabla.head(4)["Tela"].values
seleccion

<StringArray>
['Oxford.jpg', 'Terry.jpg', 'Polyester.jpg', 'Tweed.jpg']
Length: 4, dtype: str

In [21]:
for file in seleccion:
    ruta = os.path.join("img/telas", file)

    img = cv2.imread(ruta)
    img_gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    pts = encontrar_esquinas(img_gray)
    img_pts = dibujar_puntos(img.copy(), pts)

    comparacion = cv2.hconcat([img, img_pts])

    cv2.imshow(f"Original vs Esquinas - {file}", comparacion)
    cv2.waitKey(0)

cv2.destroyAllWindows()

In [22]:
tabla.to_csv("densidad_telas.csv", index=False)